# Librerías

In [267]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from imblearn.pipeline import Pipeline as ImbPipeline
import seaborn as sns
from imblearn.over_sampling import SMOTENC
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import  RobustScaler
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (f1_score,
                             recall_score, 
                             precision_score, 
                             roc_auc_score, 
                             confusion_matrix)
import scipy.stats as stats
import math
import matplotlib.gridspec as gridspec

# Limpiar datos

In [220]:
datos = pd.read_csv(filepath_or_buffer= "../../data/churn-bigml.csv", sep = ",", header=0, index_col='id_customer')
datos

,state,account_length,area_code,international_plan,voice_mail_plan,number_vmail_messages,total_day_minutes,total_day_calls,total_day_charge,total_eve_minutes,total_eve_calls,total_eve_charge,total_night_minutes,total_night_calls,total_night_charge,total_intl_minutes,total_intl_calls,total_intl_charge,customer_service_calls,churn
id_customer,,,,,,,,,,,,,,,,,,,,
1,LA,117,408,f,f,0,184.5,97,31.37,351.6,80,29.89,215.8,90,9.71,8.7,4,2.35,1,f
2,IN,65,415,f,f,0,129.1,137,21.95,228.5,83,19.42,208.8,111,9.40,12.7,6,3.43,4,t
3,NY,161,415,f,f,0,332.9,67,56.59,317.8,97,27.01,160.6,128,7.23,5.4,9,1.46,4,t
4,SC,111,415,f,f,0,110.4,103,18.77,137.3,102,11.67,189.6,105,8.53,7.7,6,2.08,2,f
5,HI,49,510,f,f,0,119.3,117,20.28,215.1,109,18.28,178.7,90,8.04,11.1,1,3.00,1,f
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3329,SC,79,415,f,f,0,134.7,98,22.90,189.7,68,16.12,221.4,128,9.96,11.8,5,3.19,2,f
3330,AZ,192,415,f,t,36,156.2,77,26.55,215.5,126,18.32,279.1,83,12.56,9.9,6,2.67,2,f
3331,WV,68,415,f,f,0,231.1,57,39.29,153.4,55,13.04,191.3,123,8.61,9.6,4,2.59,3,f


In [221]:
datos["area_code"] = datos["area_code"].astype("category")
datos["state"] = datos["state"].astype("category")
dummies = datos.select_dtypes(include = ['object']).columns.to_list()
datos = pd.get_dummies(datos, columns=dummies, drop_first=True)
datos = datos.rename(columns = {"international_plan_t":"international_plan", "voice_mail_plan_t":"voice_mail_plan", "churn_t":"churn"})
datos.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3333 entries, 1 to 3333
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype   
---  ------                  --------------  -----   
 0   state                   3333 non-null   category
 1   account_length          3333 non-null   int64   
 2   area_code               3333 non-null   category
 3   number_vmail_messages   3333 non-null   int64   
 4   total_day_minutes       3333 non-null   float64 
 5   total_day_calls         3333 non-null   int64   
 6   total_day_charge        3333 non-null   float64 
 7   total_eve_minutes       3333 non-null   float64 
 8   total_eve_calls         3333 non-null   int64   
 9   total_eve_charge        3333 non-null   float64 
 10  total_night_minutes     3333 non-null   float64 
 11  total_night_calls       3333 non-null   int64   
 12  total_night_charge      3333 non-null   float64 
 13  total_intl_minutes      3333 non-null   float64 
 14  total_intl_calls        3333 

In [222]:
llamadas = [col for col in datos.columns if 'calls' in col.lower()]
llamadas.remove('customer_service_calls')
datos['total_calls']=datos[llamadas].sum(axis=1)

cargos = [col for col in datos.columns if 'charge' in col.lower()]
datos['total_charge']=datos[cargos].sum(axis=1)

minutos = [col for col in datos.columns if 'minutes' in col.lower()]
minutos

['total_day_minutes',
 'total_eve_minutes',
 'total_night_minutes',
 'total_intl_minutes']

In [223]:
datos['customer_service_calls_>=4'] = datos['customer_service_calls']>=4
datos['total_charge_>=70'] = datos['total_charge']>=70
datos

,state,account_length,area_code,number_vmail_messages,total_day_minutes,total_day_calls,total_day_charge,total_eve_minutes,total_eve_calls,total_eve_charge,...,total_intl_calls,total_intl_charge,customer_service_calls,international_plan,voice_mail_plan,churn,total_calls,total_charge,customer_service_calls_>=4,total_charge_>=70
id_customer,,,,,,,,,,,,,,,,,,,,,
1,LA,117,408,0,184.5,97,31.37,351.6,80,29.89,...,4,2.35,1,False,False,False,271,73.32,False,True
2,IN,65,415,0,129.1,137,21.95,228.5,83,19.42,...,6,3.43,4,False,False,True,337,54.20,True,False
3,NY,161,415,0,332.9,67,56.59,317.8,97,27.01,...,9,1.46,4,False,False,True,301,92.29,True,True
4,SC,111,415,0,110.4,103,18.77,137.3,102,11.67,...,6,2.08,2,False,False,False,316,41.05,False,False
5,HI,49,510,0,119.3,117,20.28,215.1,109,18.28,...,1,3.00,1,False,False,False,317,49.60,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3329,SC,79,415,0,134.7,98,22.90,189.7,68,16.12,...,5,3.19,2,False,False,False,299,52.17,False,False
3330,AZ,192,415,36,156.2,77,26.55,215.5,126,18.32,...,6,2.67,2,False,True,False,292,60.10,False,False
3331,WV,68,415,0,231.1,57,39.29,153.4,55,13.04,...,4,2.59,3,False,False,False,239,63.53,False,False


In [224]:
columnas_excluidas = minutos + llamadas + cargos + ['number_vmail_messages', 'customer_service_calls', 'total_charge'] 
print(columnas_excluidas)
datos_limpios = datos.drop(columns = columnas_excluidas)    
datos_limpios

['total_day_minutes', 'total_eve_minutes', 'total_night_minutes', 'total_intl_minutes', 'total_day_calls', 'total_eve_calls', 'total_night_calls', 'total_intl_calls', 'total_day_charge', 'total_eve_charge', 'total_night_charge', 'total_intl_charge', 'number_vmail_messages', 'customer_service_calls', 'total_charge']


,state,account_length,area_code,international_plan,voice_mail_plan,churn,total_calls,customer_service_calls_>=4,total_charge_>=70
id_customer,,,,,,,,,
1,LA,117,408,False,False,False,271,False,True
2,IN,65,415,False,False,True,337,True,False
3,NY,161,415,False,False,True,301,True,True
4,SC,111,415,False,False,False,316,False,False
5,HI,49,510,False,False,False,317,False,False
...,...,...,...,...,...,...,...,...,...
3329,SC,79,415,False,False,False,299,False,False
3330,AZ,192,415,False,True,False,292,False,False
3331,WV,68,415,False,False,False,239,False,False


# Train/test

In [225]:
datos_limpios, churn = datos_limpios.drop(columns='churn'), datos_limpios['churn']
datos_limpios, datos_limpios_test, churn, churn_test = train_test_split(
    datos_limpios, churn,   
    test_size = .2, 
    random_state= 123, 
    stratify=churn)
datos_limpios

,state,account_length,area_code,international_plan,voice_mail_plan,total_calls,customer_service_calls_>=4,total_charge_>=70
id_customer,,,,,,,,
446,VA,121,415,False,False,357,False,False
2371,NM,93,415,False,False,291,False,True
3223,NC,72,415,False,False,338,False,False
2700,NJ,88,510,False,False,290,False,False
794,OK,96,415,False,False,252,True,True
...,...,...,...,...,...,...,...,...
2155,OK,46,415,False,False,389,False,False
1980,SD,138,510,False,False,286,False,False
240,NY,167,415,False,False,320,False,False


# Escalar datos

In [226]:
columnas_numericas = datos_limpios.select_dtypes(include = 'number').columns.tolist()
columnas_numericas

['account_length', 'total_calls']

In [227]:
escalador_datos = RobustScaler()
escalador = escalador_datos.fit(datos_limpios[columnas_numericas])
datos_trans = escalador.transform(datos_limpios[columnas_numericas])
datos_trans

array([[ 0.38888889,  1.13043478],
       [-0.12962963, -0.30434783],
       [-0.51851852,  0.7173913 ],
       ...,
       [ 1.24074074,  0.32608696],
       [-0.59259259, -0.91304348],
       [ 0.24074074,  0.89130435]])

In [228]:
datos_normalizados = pd.DataFrame(datos_trans, columns = columnas_numericas, index = datos_limpios.index)
columnas_categoricas = [col for col in datos_limpios.columns if col not in columnas_numericas]
datos_categoricos = datos_limpios[columnas_categoricas]
datos_normalizados = pd.concat([datos_categoricos, datos_normalizados], axis = 1)
datos_normalizados

,state,area_code,international_plan,voice_mail_plan,customer_service_calls_>=4,total_charge_>=70,account_length,total_calls
id_customer,,,,,,,,
446,VA,415,False,False,False,False,0.388889,1.130435
2371,NM,415,False,False,False,True,-0.129630,-0.304348
3223,NC,415,False,False,False,False,-0.518519,0.717391
2700,NJ,510,False,False,False,False,-0.222222,-0.326087
794,OK,415,False,False,True,True,-0.074074,-1.152174
...,...,...,...,...,...,...,...,...
2155,OK,415,False,False,False,False,-1.000000,1.826087
1980,SD,510,False,False,False,False,0.703704,-0.413043
240,NY,415,False,False,False,False,1.240741,0.326087


In [229]:
escalador_test = escalador_datos.fit(datos_limpios_test[columnas_numericas])
datos_trans_test = escalador_test.transform(datos_limpios_test[columnas_numericas])
datos_trans_test

array([[ 0.46601942,  0.20618557],
       [ 1.57281553,  0.6185567 ],
       [ 0.7184466 , -0.39175258],
       ...,
       [ 0.42718447,  0.        ],
       [ 0.67961165,  0.30927835],
       [-0.27184466,  1.5257732 ]])

In [230]:
datos_normalizados_test = pd.DataFrame(datos_trans_test, columns = columnas_numericas, index = datos_limpios_test.index)
columnas_categoricas = [col for col in datos_limpios_test.columns if col not in columnas_numericas]
datos_categoricos_test = datos_limpios_test[columnas_categoricas]
datos_normalizados_test = pd.concat([datos_categoricos_test, datos_normalizados_test], axis = 1)
datos_normalizados_test

,state,area_code,international_plan,voice_mail_plan,customer_service_calls_>=4,total_charge_>=70,account_length,total_calls
id_customer,,,,,,,,
2833,MN,415,True,True,False,True,0.466019,0.206186
2648,NC,415,False,False,False,True,1.572816,0.618557
727,NJ,408,False,False,False,True,0.718447,-0.391753
373,AR,510,False,False,False,False,0.291262,0.371134
48,KS,408,True,False,False,False,-0.174757,0.268041
...,...,...,...,...,...,...,...,...
511,IN,408,False,True,True,False,-0.388350,0.432990
1413,TX,415,False,False,False,False,0.563107,-0.371134
2352,MN,408,False,False,False,True,0.427184,0.000000


# Modelos

## Funciones

In [231]:
def riesgo_de_estados(df_train, y_train, df_test):
    df_temp = df_train.copy()
    df_temp['churn'] = y_train
    tasa_churn_estados = df_temp.groupby('state')['churn'].mean()
    q75 = tasa_churn_estados.quantile(0.75)
    q25 = tasa_churn_estados.quantile(0.25)
    estados_altos = tasa_churn_estados[tasa_churn_estados >= q75].index
    estados_bajos = tasa_churn_estados[tasa_churn_estados <= q25].index

    condiciones = [
        df_train['state'].isin(estados_altos),
        df_train['state'].isin(estados_bajos)
    ]
    valores = [2,0]
    df_train['state_risk'] =np.select(condiciones, valores, 1)
    df_train.drop(columns='state', inplace=True)
    condiciones = [
            df_test['state'].isin(estados_altos),
            df_test['state'].isin(estados_bajos)
    ]
    valores = [2,0]
    df_test['state_risk'] =np.select(condiciones, valores,1) 
    df_test.drop(columns = 'state', inplace=True)

## Sin oversampling

### Regresión logística

In [232]:
skf = StratifiedKFold(n_splits = 5, shuffle=True, random_state=420)

historial_resultados = []

for fold, (train_index, validation_index) in enumerate(skf.split(datos_normalizados, churn)):
    print(f'------------Iterando Fold {fold+1}------------')
    X_train = datos_normalizados.iloc[train_index].copy()
    y_train = churn.iloc[train_index].copy()
    X_validation = datos_normalizados.iloc[validation_index].copy()
    y_validation = churn.iloc[validation_index].copy()

    riesgo_de_estados(X_train, y_train, X_validation)

    X_train = pd.get_dummies(X_train, drop_first=True)
    X_validation = pd.get_dummies(X_validation, drop_first=True)

    modelo = LogisticRegression(random_state=378)
    modelo.fit(X_train, y_train)

    y_proba = modelo.predict_proba(X_validation)
    y_proba = y_proba[:,1]
    umbral = 0.2
    y_pred = [False if proba<=umbral else True for proba in y_proba]

    f1 = f1_score(y_validation, y_pred)
    recall = recall_score(y_validation, y_pred)
    precision = precision_score(y_validation, y_pred)
    roc_auc = roc_auc_score(y_validation, y_proba)

    historial_resultados.append({
        'fold': fold + 1,
        'f1_score': f1,
        'recall': recall,
        'precision': precision,
        'roc_auc': roc_auc
    })
df_resultados = pd.DataFrame(historial_resultados).set_index('fold')
display(df_resultados)
print("\nPromedios:")
print(df_resultados.mean())

------------Iterando Fold 1------------
------------Iterando Fold 2------------
------------Iterando Fold 3------------
------------Iterando Fold 4------------
------------Iterando Fold 5------------


C:\Users\mois_\AppData\Local\Temp\ipykernel_15688\2507682459.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tasa_churn_estados = df_temp.groupby('state')['churn'].mean()
C:\Users\mois_\AppData\Local\Temp\ipykernel_15688\2507682459.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tasa_churn_estados = df_temp.groupby('state')['churn'].mean()
C:\Users\mois_\AppData\Local\Temp\ipykernel_15688\2507682459.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the futu

,f1_score,recall,precision,roc_auc
fold,,,,
1,0.570000,0.730769,0.467213,0.854954
2,0.635417,0.792208,0.530435,0.875114
3,0.599034,0.805195,0.476923,0.859478
4,0.680203,0.870130,0.558333,0.907838
5,0.607843,0.805195,0.488189,0.836495



Promedios:
f1_score     0.618499
recall       0.800699
precision    0.504219
roc_auc      0.866776
dtype: float64


### Test

In [233]:
X_train = datos_normalizados.copy()
y_train = churn.copy()
X_test = datos_normalizados_test.copy()
y_test = churn_test.copy()

In [234]:
riesgo_de_estados(X_train, y_train, X_test)

X_train = pd.get_dummies(X_train, drop_first=True)
X_test = pd.get_dummies(X_test, drop_first=True)

modelo = LogisticRegression(random_state=378)
modelo.fit(X_train, y_train)

y_proba = modelo.predict_proba(X_test)
y_proba = y_proba[:,1]
umbral = 0.2
y_pred = [False if proba<=umbral else True for proba in y_proba]

print(f"F1 Score: ", f1_score(y_test, y_pred))
print(f"Recall: ", recall_score(y_test, y_pred))
print(f"Precission: ", precision_score(y_test, y_pred))
print(f"ROC, AUC: ", roc_auc_score(y_test, y_proba))
print("Matriz de confusión: ")
print(confusion_matrix(y_test, y_pred))

F1 Score:  0.5882352941176471


C:\Users\mois_\AppData\Local\Temp\ipykernel_15688\2507682459.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tasa_churn_estados = df_temp.groupby('state')['churn'].mean()


Recall:  0.7731958762886598
Precission:  0.47468354430379744
ROC, AUC:  0.8354675348164224
Matriz de confusión: 
[[487  83]
 [ 22  75]]


### Regresión logística con pesos

In [235]:
skf = StratifiedKFold(n_splits = 5, shuffle=True, random_state=420)

historial_resultados = []

for fold, (train_index, validation_index) in enumerate(skf.split(datos_normalizados, churn)):
    print(f'------------Iterando Fold {fold+1}------------')
    X_train = datos_normalizados.iloc[train_index].copy()
    y_train = churn.iloc[train_index].copy()
    X_validation = datos_normalizados.iloc[validation_index].copy()
    y_validation = churn.iloc[validation_index].copy()

    riesgo_de_estados(X_train, y_train, X_validation)

    X_train = pd.get_dummies(X_train, drop_first=True)
    X_validation = pd.get_dummies(X_validation, drop_first=True)

    modelo = LogisticRegression(random_state=378, class_weight='balanced')
    modelo.fit(X_train, y_train)

    y_proba = modelo.predict_proba(X_validation)
    y_pred = modelo.predict(X_validation)

    f1 = f1_score(y_validation, y_pred)
    recall = recall_score(y_validation, y_pred)
    precision = precision_score(y_validation, y_pred)
    roc_auc = roc_auc_score(y_validation, y_proba[:,1])

    historial_resultados.append({
        'fold': fold + 1,
        'f1_score': f1,
        'recall': recall,
        'precision': precision,
        'roc_auc': roc_auc
    })
df_resultados = pd.DataFrame(historial_resultados).set_index('fold')
display(df_resultados)
print("\nPromedios:")
print(df_resultados.mean())

------------Iterando Fold 1------------


C:\Users\mois_\AppData\Local\Temp\ipykernel_15688\2507682459.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tasa_churn_estados = df_temp.groupby('state')['churn'].mean()


------------Iterando Fold 2------------
------------Iterando Fold 3------------
------------Iterando Fold 4------------
------------Iterando Fold 5------------


C:\Users\mois_\AppData\Local\Temp\ipykernel_15688\2507682459.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tasa_churn_estados = df_temp.groupby('state')['churn'].mean()
C:\Users\mois_\AppData\Local\Temp\ipykernel_15688\2507682459.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tasa_churn_estados = df_temp.groupby('state')['churn'].mean()
C:\Users\mois_\AppData\Local\Temp\ipykernel_15688\2507682459.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the futu

,f1_score,recall,precision,roc_auc
fold,,,,
1,0.597403,0.884615,0.450980,0.856388
2,0.610329,0.844156,0.477941,0.869589
3,0.587719,0.870130,0.443709,0.862326
4,0.627273,0.896104,0.482517,0.910686
5,0.577778,0.844156,0.439189,0.838232



Promedios:
f1_score     0.600100
recall       0.867832
precision    0.458867
roc_auc      0.867444
dtype: float64


### Test

In [236]:
X_train = datos_normalizados.copy()
y_train = churn.copy()
X_test = datos_normalizados_test.copy()
y_test = churn_test.copy()

In [237]:
riesgo_de_estados(X_train, y_train, X_test)

X_train = pd.get_dummies(X_train, drop_first=True)
X_test = pd.get_dummies(X_test, drop_first=True)

modelo = LogisticRegression(random_state=378, class_weight='balanced')
modelo.fit(X_train, y_train)

y_pred = modelo.predict(X_test)
y_proba = modelo.predict_proba(X_test)

print(f"F1 Score: ", f1_score(y_test, y_pred))
print(f"Recall: ", recall_score(y_test, y_pred))
print(f"Precission: ", precision_score(y_test, y_pred))
print(f"ROC, AUC: ", roc_auc_score(y_test, y_proba[:,1]))
print("Matriz de confusión: ")
print(confusion_matrix(y_test, y_pred))

F1 Score:  0.54
Recall:  0.8350515463917526
Precission:  0.39901477832512317
ROC, AUC:  0.8368963646228974
Matriz de confusión: 
[[448 122]
 [ 16  81]]


C:\Users\mois_\AppData\Local\Temp\ipykernel_15688\2507682459.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tasa_churn_estados = df_temp.groupby('state')['churn'].mean()


## Oversampling

### Regresión logística

In [248]:
skf = StratifiedKFold(n_splits = 5, shuffle=True, random_state=420)

historial_resultados = []

for fold, (train_index, validation_index) in enumerate(skf.split(datos_normalizados, churn)):
    print(f'------------Iterando Fold {fold+1}------------')

    X_train = datos_normalizados.iloc[train_index].copy()
    y_train = churn.iloc[train_index].copy()
    X_validation = datos_normalizados.iloc[validation_index].copy()
    y_validation = churn.iloc[validation_index].copy()

    sm = SMOTENC(categorical_features=columnas_categoricas,random_state=420, sampling_strategy=0.3)
    X_train_res, y_train_res = sm.fit_resample(datos_normalizados, churn)    

    riesgo_de_estados(X_train_res, y_train_res, X_validation)

    X_train_res = pd.get_dummies(X_train_res, drop_first=True)
    X_validation = pd.get_dummies(X_validation, drop_first=True)

    modelo = LogisticRegression(random_state=378)
    modelo.fit(X_train_res, y_train_res)

    y_proba = modelo.predict_proba(X_validation)
    y_proba = y_proba[:,1]
    umbral = 0.3
    y_pred = [False if proba<=umbral else True for proba in y_proba]

    f1 = f1_score(y_validation, y_pred)
    recall = recall_score(y_validation, y_pred)
    precision = precision_score(y_validation, y_pred)
    roc_auc = roc_auc_score(y_validation, y_proba)

    historial_resultados.append({
        'fold': fold + 1,
        'f1_score': f1,
        'recall': recall,
        'precision': precision,
        'roc_auc': roc_auc
    })
df_resultados = pd.DataFrame(historial_resultados).set_index('fold')
display(df_resultados)
print("\nPromedios:")
print(df_resultados.mean())

------------Iterando Fold 1------------


C:\Users\mois_\AppData\Local\Temp\ipykernel_15688\2507682459.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tasa_churn_estados = df_temp.groupby('state')['churn'].mean()
C:\Users\mois_\AppData\Local\Temp\ipykernel_15688\2507682459.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tasa_churn_estados = df_temp.groupby('state')['churn'].mean()


------------Iterando Fold 2------------
------------Iterando Fold 3------------


C:\Users\mois_\AppData\Local\Temp\ipykernel_15688\2507682459.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tasa_churn_estados = df_temp.groupby('state')['churn'].mean()
C:\Users\mois_\AppData\Local\Temp\ipykernel_15688\2507682459.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tasa_churn_estados = df_temp.groupby('state')['churn'].mean()


------------Iterando Fold 4------------
------------Iterando Fold 5------------


C:\Users\mois_\AppData\Local\Temp\ipykernel_15688\2507682459.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tasa_churn_estados = df_temp.groupby('state')['churn'].mean()


,f1_score,recall,precision,roc_auc
fold,,,,
1,0.584158,0.756410,0.475806,0.874944
2,0.646154,0.818182,0.533898,0.892458
3,0.611650,0.818182,0.488372,0.878161
4,0.683168,0.896104,0.552000,0.919799
5,0.631068,0.844156,0.503876,0.855690



Promedios:
f1_score     0.631240
recall       0.826607
precision    0.510791
roc_auc      0.884211
dtype: float64


### Test

In [255]:
sm = SMOTENC(categorical_features=columnas_categoricas,random_state=420, sampling_strategy=0.3)
X_train_res, y_train_res = sm.fit_resample(datos_normalizados, churn)
y_train_res.value_counts(normalize=True)

churn
False    0.769231
True     0.230769
Name: proportion, dtype: float64

In [256]:
X_train = X_train_res.copy()
y_train = y_train_res.copy()
X_test = datos_normalizados_test.copy()
y_test = churn_test.copy()

riesgo_de_estados(X_train, y_train, X_test)

X_train = pd.get_dummies(X_train, drop_first=True)
X_test = pd.get_dummies(X_test, drop_first=True)

modelo = LogisticRegression(random_state=378)
modelo.fit(X_train, y_train)

y_proba = modelo.predict_proba(X_test)
y_proba = y_proba[:,1]
umbral = 0.3
y_pred = [False if proba<=umbral else True for proba in y_proba]

print(f"F1 Score: ", f1_score(y_test, y_pred))
print(f"Recall: ", recall_score(y_test, y_pred))
print(f"Precission: ", precision_score(y_test, y_pred))
print(f"ROC, AUC: ", roc_auc_score(y_test, y_proba))
print("Matriz de confusión: ")
print(confusion_matrix(y_test, y_pred))

F1 Score:  0.5826771653543307
Recall:  0.7628865979381443
Precission:  0.4713375796178344
ROC, AUC:  0.8396816784228613
Matriz de confusión: 
[[487  83]
 [ 23  74]]


C:\Users\mois_\AppData\Local\Temp\ipykernel_15688\2507682459.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tasa_churn_estados = df_temp.groupby('state')['churn'].mean()


### Regresión con pesos

In [240]:
skf = StratifiedKFold(n_splits = 5, shuffle=True, random_state=420)

historial_resultados = []

for fold, (train_index, validation_index) in enumerate(skf.split(datos_normalizados, churn)):
    print(f'------------Iterando Fold {fold+1}------------')

    X_train = datos_normalizados.iloc[train_index].copy()
    y_train = churn.iloc[train_index].copy()
    X_validation = datos_normalizados.iloc[validation_index].copy()
    y_validation = churn.iloc[validation_index].copy()

    sm = SMOTENC(categorical_features=columnas_categoricas,random_state=420, sampling_strategy=0.3)
    X_train_res, y_train_res = sm.fit_resample(datos_normalizados, churn)    

    riesgo_de_estados(X_train_res, y_train_res, X_validation)

    X_train_res = pd.get_dummies(X_train_res, drop_first=True)
    X_validation = pd.get_dummies(X_validation, drop_first=True)

    modelo = LogisticRegression(random_state=378, class_weight='balanced')
    modelo.fit(X_train_res, y_train_res)

    y_proba = modelo.predict_proba(X_validation)
    y_pred = modelo.predict(X_validation)

    f1 = f1_score(y_validation, y_pred)
    recall = recall_score(y_validation, y_pred)
    precision = precision_score(y_validation, y_pred)
    roc_auc = roc_auc_score(y_validation, y_proba[:,1])

    historial_resultados.append({
        'fold': fold + 1,
        'f1_score': f1,
        'recall': recall,
        'precision': precision,
        'roc_auc': roc_auc
    })
df_resultados = pd.DataFrame(historial_resultados).set_index('fold')
display(df_resultados)
print("\nPromedios:")
print(df_resultados.mean())

------------Iterando Fold 1------------
------------Iterando Fold 2------------


C:\Users\mois_\AppData\Local\Temp\ipykernel_15688\2507682459.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tasa_churn_estados = df_temp.groupby('state')['churn'].mean()
C:\Users\mois_\AppData\Local\Temp\ipykernel_15688\2507682459.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tasa_churn_estados = df_temp.groupby('state')['churn'].mean()


------------Iterando Fold 3------------
------------Iterando Fold 4------------


C:\Users\mois_\AppData\Local\Temp\ipykernel_15688\2507682459.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tasa_churn_estados = df_temp.groupby('state')['churn'].mean()
C:\Users\mois_\AppData\Local\Temp\ipykernel_15688\2507682459.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tasa_churn_estados = df_temp.groupby('state')['churn'].mean()


------------Iterando Fold 5------------


C:\Users\mois_\AppData\Local\Temp\ipykernel_15688\2507682459.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tasa_churn_estados = df_temp.groupby('state')['churn'].mean()


,f1_score,recall,precision,roc_auc
fold,,,,
1,0.592920,0.858974,0.452703,0.874916
2,0.622642,0.857143,0.488889,0.889440
3,0.592920,0.870130,0.449664,0.879585
4,0.642202,0.909091,0.496454,0.923046
5,0.584071,0.857143,0.442953,0.855235



Promedios:
f1_score     0.606951
recall       0.870496
precision    0.466133
roc_auc      0.884444
dtype: float64


### Test

In [260]:
sm = SMOTENC(categorical_features=columnas_categoricas,random_state=420, sampling_strategy=0.3)
X_train_res, y_train_res = sm.fit_resample(datos_normalizados, churn)
y_train_res.value_counts(normalize=True)

churn
False    0.769231
True     0.230769
Name: proportion, dtype: float64

In [261]:
X_train = X_train_res.copy()
y_train = y_train_res.copy()
X_test = datos_normalizados_test.copy()
y_test = churn_test.copy()

riesgo_de_estados(X_train, y_train, X_test)

X_train = pd.get_dummies(X_train, drop_first=True)
X_test = pd.get_dummies(X_test, drop_first=True)

modelo = LogisticRegression(random_state=378, class_weight='balanced')
modelo.fit(X_train, y_train)

y_proba = modelo.predict_proba(X_test)[:,1]
y_pred = modelo.predict(X_test)

print(f"F1 Score: ", f1_score(y_test, y_pred))
print(f"Recall: ", recall_score(y_test, y_pred))
print(f"Precission: ", precision_score(y_test, y_pred))
print(f"ROC, AUC: ", roc_auc_score(y_test, y_proba))
print("Matriz de confusión: ")
print(confusion_matrix(y_test, y_pred))

F1 Score:  0.5436241610738255
Recall:  0.8350515463917526
Precission:  0.40298507462686567
ROC, AUC:  0.840332790739736
Matriz de confusión: 
[[450 120]
 [ 16  81]]


C:\Users\mois_\AppData\Local\Temp\ipykernel_15688\2507682459.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tasa_churn_estados = df_temp.groupby('state')['churn'].mean()
